# Thế hệ 4 — Coupled Earth System Models / CMIP (CMIP3 → CMIP6)

CMIP ghép GCM khí quyển với đại dương, băng biển, đất, chu trình carbon thành **Earth System Model**
đầy đủ, và chạy **ensemble đa mô hình** (nhiều trung tâm nghiên cứu, nhiều điều kiện ban đầu) để định
lượng độ bất định — đúng cách bài báo #2 (CORDEX-SEA) dùng 11 GCM CMIP6 khác nhau làm input downscale.

**Lưu ý minh bạch**: tải thật dữ liệu CMIP6 archive (ESGF) cần đăng ký/API phức tạp, không khả thi
trong demo nhanh này. Notebook dưới đây dùng ĐÚNG 1 lát dữ liệu OISST thật làm "quan trắc tham chiếu",
rồi tạo **ensemble minh hoạ** bằng cách nhiễu loạn thống kê (bootstrap + nhiễu Gaussian nhỏ) để mô
phỏng khái niệm "nhiều thành viên ensemble quanh 1 giá trị tham chiếu" — đây là **dữ liệu mẫu minh
hoạ cho khái niệm ensemble, KHÔNG phải ensemble CMIP6 thật**.

In [1]:
DATA_PATH = r"/Users/tuannghiat/Downloads/USTH Phenikaa - Climate Modelling /08_The_He_Mo_Hinh_Thoi_Tiet/_data_cache/oisst_SEA_2020_monthly_sample.nc"
print("Dùng dữ liệu thật:", DATA_PATH)

Dùng dữ liệu thật: /Users/tuannghiat/Downloads/USTH Phenikaa - Climate Modelling /08_The_He_Mo_Hinh_Thoi_Tiet/_data_cache/oisst_SEA_2020_monthly_sample.nc


In [2]:
import xarray as xr
import numpy as np

ds = xr.open_dataset(DATA_PATH)
sst_series = ds['sst'].mean(dim=['lat', 'lon']).squeeze().values  # SST thật, trung bình vùng, 12 tháng 2020
print("Chuỗi SST thật (quan trắc tham chiếu, OISST 2020):", np.round(sst_series, 3))

rng = np.random.default_rng(42)
n_members = 11  # khớp số GCM CMIP6 dùng trong bài báo #2 (CORDEX-SEA)
# Mỗi "thành viên ensemble minh hoạ" = quan trắc thật + độ lệch hệ thống (bias) ngẫu nhiên nhỏ mô phỏng
# chênh lệch giữa các trung tâm mô hình khí hậu khác nhau — ĐÂY LÀ DỮ LIỆU MẪU MINH HOẠ, KHÔNG PHẢI
# OUTPUT THẬT CỦA 11 GCM CMIP6.
biases = rng.normal(loc=0.0, scale=0.6, size=n_members)   # lệch trung bình mỗi "mô hình" (độ C)
noise_scale = rng.uniform(0.2, 0.5, size=n_members)       # biến động nội mùa khác nhau mỗi "mô hình"

ensemble = np.array([
    sst_series + biases[i] + rng.normal(0, noise_scale[i], size=sst_series.shape)
    for i in range(n_members)
])
print("\nEnsemble spread (độ lệch chuẩn giữa 11 'thành viên' minh hoạ) theo từng tháng:")
print(np.round(ensemble.std(axis=0), 3))
print("\nTrung bình ensemble so với quan trắc thật (sai khác trung bình):",
      round(float(np.mean(ensemble.mean(axis=0) - sst_series)), 4), "độ C")
print("\n⚠️ Nhắc lại: 11 thành viên trên là nhiễu thống kê minh hoạ khái niệm multi-model ensemble,")
print("không phải dữ liệu CMIP6 thật — muốn dữ liệu thật phải qua ESGF/Copernicus CDS với đăng ký.")

Chuỗi SST thật (quan trắc tham chiếu, OISST 2020): [25.896 25.453 26.053 27.949 28.372 30.117 29.964 30.117 29.897 29.452
 27.671 27.205]

Ensemble spread (độ lệch chuẩn giữa 11 'thành viên' minh hoạ) theo từng tháng:
[0.713 0.706 0.61  0.564 0.681 0.629 0.607 0.598 0.635 0.563 0.513 0.674]

Trung bình ensemble so với quan trắc thật (sai khác trung bình): -0.1486 độ C

⚠️ Nhắc lại: 11 thành viên trên là nhiễu thống kê minh hoạ khái niệm multi-model ensemble,
không phải dữ liệu CMIP6 thật — muốn dữ liệu thật phải qua ESGF/Copernicus CDS với đăng ký.
